In [ ]:
import json
import logging
import os
from datetime import datetime as dt, timezone as tz

from pydantic import BaseModel, TypeAdapter

from dotenv import load_dotenv
load_dotenv()

from sqlalchemy import create_engine, insert, select, delete, MetaData
from sqlalchemy.engine import URL

from db_types import Catalog, Chart, Player, Relationship, Score

In [ ]:
# function_app.py

_HASH_LEN = 16
def cid(q: Union[int, str]) -> Union[int, str]:
    if isinstance(q, str) and len(q) == _HASH_LEN:
        # It's a hash
        return q
    chart_id_int = soft_int(q)
    if isinstance(chart_id_int, int):
        # It's an index
        return chart_id_int
    else:
        # It's neither...and probably bad
        return q
    

def dt2str(d: Dict) -> Dict:
    r = {}
    for k in d:
        if isinstance(d[k], dt):
            r[k] = d[k].isoformat()
        else:
            r[k] = d[k]
    return r


def soft_int(s: str) -> Union[int, str]:
    try:
        return int(s)
    except:
        return s

# Database connection
def engine_construct():
    url_construct = URL.create(
        "mssql+pyodbc",
        username=os.environ["SCOBILITY_UID"],
        password=os.environ["SCOBILITY_PWD"],
        host=os.environ["SCOBILITY_SERVER"],
        database=os.environ["SCOBILITY_DATABASE"],
        query={"driver": "ODBC Driver 18 for SQL Server"}
    )

    return create_engine(url_construct)

_ENGINE = engine_construct()
_MD = MetaData()
_MD.reflect(bind=_ENGINE)


In [ ]:
import glob
from typing import Any

scrape = sorted(glob.glob(r"C:\Users\telpi\OneDrive\Documents\GitHub\scobility\scratch\scobility_itl2026_*.json"))[-1]
spice_calc_time = os.path.getmtime(scrape)
print(scrape)

with open(scrape, "r") as f:
    tourney = json.load(f)


In [ ]:
from scratch.scobility import Clear

def upgrade_chart(c: dict[str, Any]) -> Chart:
    c["catalog_id"] = 6
    c["global_chart_id"] = 6000 + int(c["s_id"])
    c["chart_id"] = c["s_id"]
    c["spice_calc_time"] = spice_calc_time
    return TypeAdapter(Chart).validate_python(c)

def upgrade_player(p: dict[str, Any]) -> Player:
    p["catalog_id"] = 6
    p["performance_id"] = 60000 + int(p["e_id"])
    p["entrant_id"] = p["e_id"]
    p["groovestats_id"] = p["g_id"]
    p["boogiestats_id"] = None
    p["scobility_calc_time"] = spice_calc_time
    return TypeAdapter(Player).validate_python(p)

def upgrade_score(s: dict[str, Any], i: int) -> Score:
    s["catalog_id"] = 6
    s["score_id"] = 6000000 + i
    s["global_chart_id"] = 6000 + int(s["s_id"])
    s["performance_id"] = 60000 + int(s["e_id"])
    s["score"] = 1 - s["value"]
    s["prediction"] = 1
    s["prediction_calc_time"] = spice_calc_time
    s["clear_int"] = s.get("clear_int", s["clear"])
    s["clear"] = Clear(s["clear_int"]).name
    return TypeAdapter(Score).validate_python(s)

charts_db = [upgrade_chart(c) for c in tourney["songs"]]
players_db = [upgrade_player(p) for p in tourney["players"]]
scores_db = [upgrade_score(s, i) for i, s in enumerate(tourney["scores"])]

for c in charts_db[:3]:
    print(c)
for p in players_db[:3]:
    print(p)
for s in scores_db[:3]:
    print(s)


In [ ]:
# clear all catalog 6 data

with _ENGINE.connect() as connection:
    result = connection.execute(
        delete(_MD.tables['Score'])
        .where(_MD.tables['Score'].c.catalog_id == 6)
    )
    print(result)

    result = connection.execute(
        delete(_MD.tables['Player'])
        .where(_MD.tables['Player'].c.catalog_id == 6)
    )
    print(result)

    result = connection.execute(
        delete(_MD.tables['Chart'])
        .where(_MD.tables['Chart'].c.catalog_id == 6)
    )
    print(result)

    # connection.commit()

In [ ]:
# update all catalog 6 data
_STEP = 1000

with _ENGINE.connect() as connection:
    result = connection.execute(
        insert(_MD.tables['Chart']),
        [c.model_dump() for c in charts_db]
    )
    print(result)

    result = connection.execute(
        insert(_MD.tables['Player']),
        [p.model_dump() for p in players_db]
    )
    print(result)
    
    i = 0
    while i < len(scores_db):
        result = connection.execute(
            insert(_MD.tables['Score']),
            [s.model_dump() for s in scores_db[i:i+_STEP]]
        )
        print(f"{i:7d}~{i + _STEP:7d} of {len(scores_db):7d}: {result}")
        i += _STEP
    
    # connection.commit()